# Chorus Visualization

Stage 3 chorus sanity plots: LFO shape, modulated delay time, sustained-sine modulation, and wet-path HF rolloff. The equations mirror `Source/DSP/ChorusConfig.h` and `Source/DSP/Chorus.cpp` for visual inspection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample_rate = 48_000
center_delay_ms = 7.0
modulation_range_ms = 4.0
wet_lowpass_hz = 6_000.0

def one_pole_lowpass(signal, cutoff_hz, sample_rate):
    coefficient = 1.0 - np.exp(-2.0 * np.pi * cutoff_hz / sample_rate)
    state = 0.0
    output = np.zeros_like(signal)
    for index, sample in enumerate(signal):
        state += coefficient * (sample - state)
        output[index] = state
    return output

def fractional_delay(signal, delay_samples):
    output = np.zeros_like(signal)
    for index, delay in enumerate(delay_samples):
        read = index - delay
        if read <= 0:
            continue
        lower = int(np.floor(read))
        frac = read - lower
        if lower + 1 < len(signal):
            output[index] = (1.0 - frac) * signal[lower] + frac * signal[lower + 1]
    return output

In [ ]:
duration = 4.0
time = np.arange(int(sample_rate * duration)) / sample_rate

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for rate_hz in [0.1, 0.8, 2.0, 10.0]:
    lfo = np.sin(2.0 * np.pi * rate_hz * time)
    axes[0].plot(time, lfo, label=f'{rate_hz:g} Hz')
    delay_ms = center_delay_ms + modulation_range_ms * lfo
    axes[1].plot(time, delay_ms, label=f'{rate_hz:g} Hz')

axes[0].set_title('Sine LFO output')
axes[0].set_ylabel('LFO')
axes[0].legend()
axes[1].set_title('Full-depth delay time')
axes[1].set_ylabel('Delay (ms)')
axes[1].set_xlabel('Time (s)')
axes[1].legend()
plt.tight_layout()

In [ ]:
duration = 2.0
time = np.arange(int(sample_rate * duration)) / sample_rate
source = 0.5 * np.sin(2.0 * np.pi * 220.0 * time)
lfo_left = np.sin(2.0 * np.pi * 0.8 * time)
lfo_right = np.sin(2.0 * np.pi * 0.8 * time + np.pi)
center_samples = round(center_delay_ms * sample_rate / 1000.0)
range_samples = modulation_range_ms * sample_rate / 1000.0
wet_left = one_pole_lowpass(fractional_delay(source, center_samples + range_samples * lfo_left), wet_lowpass_hz, sample_rate)
wet_right = one_pole_lowpass(fractional_delay(source, center_samples + range_samples * lfo_right), wet_lowpass_hz, sample_rate)

start = int(0.9 * sample_rate)
stop = start + int(0.04 * sample_rate)
plt.figure(figsize=(10, 4))
plt.plot(time[start:stop], wet_left[start:stop], label='Left wet')
plt.plot(time[start:stop], wet_right[start:stop], label='Right wet', alpha=0.8)
plt.title('Sustained sine through decorrelated wet voices')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()

In [ ]:
rng = np.random.default_rng(1234)
noise = rng.normal(0.0, 0.35, sample_rate * 2)
filtered = one_pole_lowpass(noise, wet_lowpass_hz, sample_rate)
window = np.hanning(len(noise))
freqs = np.fft.rfftfreq(len(noise), 1.0 / sample_rate)
noise_spectrum = np.abs(np.fft.rfft(noise * window))
filtered_spectrum = np.abs(np.fft.rfft(filtered * window))
response_db = 20.0 * np.log10(np.maximum(filtered_spectrum, 1e-12) / np.maximum(noise_spectrum, 1e-12))

plt.figure(figsize=(10, 4))
plt.semilogx(freqs[1:], response_db[1:])
plt.axvline(wet_lowpass_hz, color='r', linestyle='--', label='6 kHz target')
plt.axhline(-3.0, color='k', linestyle=':', label='-3 dB')
plt.ylim(-24, 3)
plt.xlim(50, 20_000)
plt.title('Wet-path one-pole low-pass response on white noise')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Relative level (dB)')
plt.legend()
plt.tight_layout()